# 09 · Aprendizaje No Supervisado

**Objetivo:** complementar el modelo supervisado con técnicas no supervisadas para:
1. **Entender la estructura** de los datos sin usar las etiquetas
2. **Evaluar la separabilidad** de los fallos en el espacio de features
3. **Detectar anomalías** sin etiquetas — útil cuando aparecen tipos de fallo nuevos

### ¿Por qué no supervisado si ya tenemos etiquetas?
- **Visualización**: PCA y t-SNE revelan si los datos tienen estructura inherente y si los fallos son compactos o dispersos
- **Clustering**: prueba si los 5 tipos de fallo forman grupos naturales (sin supervisión)
- **IsolationForest**: detecta anomalías sin necesitar etiquetas históricas → operativo desde día 1

### Tres análisis
| Técnica | Tipo | Pregunta que responde |
|---|---|---|
| PCA + t-SNE | Reducción dimensional | ¿Son los fallos separables en 2D? ¿Qué features dominan? |
| KMeans | Clustering | ¿Los algoritmos recuperan los 5 tipos de fallo sin etiquetas? |
| IsolationForest | Detección de anomalías | ¿Cuánto peor que el supervisado? ¿Compensa la falta de etiquetas? |

In [ ]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (silhouette_score, adjusted_rand_score,
                              precision_recall_curve,
                              f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler
from utils import load_ai4i, FEATURE_COLS, FEAT_NAMES

failure_types = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
labels_map    = {0: 'Normal', 1: 'TWF', 2: 'HDF', 3: 'PWF', 4: 'OSF', 5: 'RNF'}

ai4i = load_ai4i('../data/raw/ai4i2020.csv')

X = ai4i[FEATURE_COLS].values
y = ai4i['Machine failure'].values
ftype = np.zeros(len(ai4i), dtype=int)
for i, ft in enumerate(failure_types, 1):
    ftype[ai4i[ft] == 1] = i

X_scaled = StandardScaler().fit_transform(X)

with open('../data/processed/models.pkl', 'rb') as f: models = pickle.load(f)
with open('../data/processed/splits.pkl', 'rb') as f: splits = pickle.load(f)
X_train, X_test, y_train, y_test = splits['ai4i']

print('Total muestras:', len(ai4i))
print('Features:', len(FEATURE_COLS), FEATURE_COLS)
print('Fallos por tipo:')
for ft in failure_types:
    print(f'  {ft}: {ai4i[ft].sum()}')


---
## 1. PCA — ¿son los fallos linealmente separables?

PCA (Principal Component Analysis) proyecta los datos en las direcciones de máxima varianza.
Con 2 componentes, podemos visualizar el dataset en 2D.

**¿Qué buscamos?**
- Si los fallos forman una zona claramente separada → el problema es linealmente fácil
- Si los fallos están mezclados con normales → necesitamos modelos no lineales (lo que ya tenemos)
- Los **loadings** (contribución de cada feature a cada PC) revelan qué variables dominan el espacio

> PCA no usa las etiquetas — la separación que veamos es puramente geométrica en el espacio de features.

In [ ]:
from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Normal vs Fallo
ax = axes[0]
palette = {0: '#aec6cf', 1: '#e74c3c'}
for label, name, alpha in [(0, 'Normal', 0.2), (1, 'Fallo', 0.8)]:
    mask = y == label
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=palette[label], label=name, alpha=alpha, s=8)
ax.set_title(f'PCA — Normal vs Fallo\n(varianza explicada: {pca.explained_variance_ratio_.sum():.1%})')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.legend()

# Tipos de fallo
ax = axes[1]
colors = ['#aec6cf', '#e74c3c', '#f39c12', '#27ae60', '#8e44ad', '#2980b9']
labels_map = {0: 'Normal', 1: 'TWF', 2: 'HDF', 3: 'PWF', 4: 'OSF', 5: 'RNF'}
for i in range(6):
    mask = ftype == i
    alpha = 0.15 if i == 0 else 0.9
    s = 8 if i == 0 else 25
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=colors[i], label=labels_map[i], alpha=alpha, s=s)
ax.set_title('PCA — Tipos de fallo')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.legend(markerscale=2, fontsize=8)

plt.tight_layout()
plt.show()

# Contribución de features a cada PC
loadings = pd.DataFrame(pca.components_.T, index=feat_names, columns=['PC1', 'PC2'])
print('\nLoadings PCA:')
print(loadings.round(3).to_string())


## 2. t-SNE — proyección no lineal

t-SNE (t-distributed Stochastic Neighbor Embedding) preserva las distancias locales entre puntos:
grupos que son similares en el espacio original aparecen juntos en 2D.

**Diferencia con PCA:**
- PCA preserva varianza global (distancias grandes)
- t-SNE preserva vecindades locales (grupos compactos)
- t-SNE no tiene proyección lineal → no hay "loadings" interpretables

**Parámetro `perplexity`:** controla el número de vecinos que considera cada punto (~30 es el valor típico).

> Si los fallos aparecen como clusters separados en t-SNE pero no en PCA,
> la separabilidad es **no lineal** — lo que explica por qué los modelos de árboles funcionan bien aquí.

In [ ]:
# t-SNE: proyección no lineal (alternativa a UMAP, incluida en sklearn)
print('Calculando t-SNE (puede tardar ~30s)...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_jobs=-1)
X_tsne = tsne.fit_transform(X_scaled)
print('t-SNE listo.')

palette = {0: '#aec6cf', 1: '#e74c3c'}
colors  = ['#aec6cf','#e74c3c','#f39c12','#27ae60','#8e44ad','#2980b9']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, color_by, title in [
    (axes[0], y,     't-SNE — Normal vs Fallo'),
    (axes[1], ftype, 't-SNE — Tipos de fallo'),
]:
    if color_by is y:
        for label, name, alpha in [(0,'Normal',0.15),(1,'Fallo',0.9)]:
            mask = color_by == label
            ax.scatter(X_tsne[mask,0], X_tsne[mask,1],
                       c=palette[label], label=name, alpha=alpha, s=8)
    else:
        for i in range(6):
            mask = color_by == i
            ax.scatter(X_tsne[mask,0], X_tsne[mask,1],
                       c=colors[i], label=labels_map[i],
                       alpha=0.15 if i==0 else 0.9,
                       s=8 if i==0 else 30)
    ax.set_title(title)
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    ax.legend(markerscale=2, fontsize=8)

plt.tight_layout()
plt.show()


---
## 3. Clustering — ¿recuperamos los tipos de fallo sin etiquetas?

Aplicamos KMeans **solo sobre las 339 muestras con fallo** para ver si los algoritmos
recuperan los 5 tipos (TWF, HDF, PWF, OSF, RNF) de forma natural.

**Métricas de evaluación:**
- **Silhouette score** (0 a 1): mide si las muestras están bien asignadas a su cluster vs al vecino más cercano. No requiere etiquetas verdaderas.
- **ARI** (Adjusted Rand Index, -1 a 1): mide el solapamiento entre los clusters encontrados y los tipos reales. ARI=1 significa coincidencia perfecta.

**¿Qué esperamos?**
Los 5 tipos de fallo están determinados por reglas físicas distintas. Si esas reglas
crean "zonas" compactas en el espacio de features, KMeans debería encontrarlas.
Si no (porque los tipos se solapan o el RNF es aleatorio), el ARI será bajo.

In [ ]:
X_fail_tsne = X_tsne[mask_fail]
# Solo fallos
mask_fail = y == 1
X_fail = X_scaled[mask_fail]
ftype_fail = ftype[mask_fail]
X_fail_tsne = X_tsne[mask_fail]

print(f'Muestras con fallo: {mask_fail.sum()}')
print('Distribución real:', {labels_map[i]: (ftype_fail==i).sum() for i in range(1,6)})

# KMeans k=5
km = KMeans(n_clusters=5, random_state=42, n_init=20)
km_labels = km.fit_predict(X_fail)

# Métricas de clustering
sil = silhouette_score(X_fail, km_labels)
ari = adjusted_rand_score(ftype_fail, km_labels)
print(f'\nKMeans k=5:')
print(f'  Silhouette:  {sil:.4f}  (1=perfecto, 0=solapado, <0=mal asignado)')
print(f'  ARI vs real: {ari:.4f}  (1=coincide con tipos reales, 0=aleatorio)')

# Buscar k óptimo
sil_scores = []
ks = range(2, 8)
for k in ks:
    lbl = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(X_fail)
    sil_scores.append(silhouette_score(X_fail, lbl))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Silhouette vs k
axes[0].plot(list(ks), sil_scores, 'o-', color='steelblue', lw=2)
axes[0].axvline(5, color='red', ls='--', label='k=5 (tipos reales)')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Silhouette')
axes[0].set_title('Silhouette vs número de clusters')
axes[0].legend()

# Clusters en UMAP
ax = axes[1]
cmap = plt.cm.get_cmap('tab10', 5)
for c in range(5):
    mask_c = km_labels == c
    ax.scatter(X_fail_tsne[mask_c, 0], X_fail_tsne[mask_c, 1],
               color=cmap(c), label=f'Cluster {c}', s=50, alpha=0.8)
ax.set_title('KMeans k=5 sobre fallos (espacio UMAP)')
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ¿Qué tipo de fallo predomina en cada cluster?
df_clust = pd.DataFrame({
    'cluster': km_labels,
    'tipo_real': [labels_map[t] for t in ftype_fail]
})
ct = pd.crosstab(df_clust['cluster'], df_clust['tipo_real'])
print('Tabla cluster × tipo de fallo real:')
print(ct.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
ct.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white')
ax.set_title('Composición de cada cluster KMeans')
ax.set_xlabel('Cluster'); ax.set_ylabel('Nº muestras')
ax.legend(title='Tipo fallo', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()


---
## 4. Detección de anomalías — IsolationForest vs Stacking supervisado

IsolationForest detecta anomalías **sin necesitar etiquetas de fallo**:
entrena solo con datos normales y marca como anómalas las observaciones que son
"fáciles de aislar" en árboles aleatorios (las que están en regiones poco densas).

**Caso de uso real:** en el día 1 de operación, antes de tener historial de fallos,
IsolationForest puede usarse directamente. Conforme se acumulan fallos reales,
se reemplaza por un modelo supervisado.

**Parámetro `contamination`:** proporción esperada de anomalías. Probamos distintos valores
alrededor de la tasa real de fallos (3.4%).

**Comparación con el stacking:** mide cuánto "coste" tiene no tener etiquetas.

In [ ]:
from sklearn.metrics import precision_recall_curve

# Entrenamos solo con datos normales del train set
X_train_normal = X_train[y_train == 0]
print(f'Entrenando IsolationForest solo con {len(X_train_normal)} muestras normales...')

# Buscar mejor contamination
contam_real = y_train.mean()  # tasa real de fallos
results_if = {}

for contam in [contam_real, contam_real * 0.5, contam_real * 2, 0.05]:
    iso = IsolationForest(contamination=contam, random_state=42, n_jobs=-1)
    iso.fit(X_train_normal)
    # -1 = anomalía, 1 = normal → convertir a 0/1
    y_pred_iso = (iso.predict(X_test) == -1).astype(int)
    y_score_iso = -iso.score_samples(X_test)  # mayor = más anómalo
    results_if[f'contam={contam:.3f}'] = {
        'Precision': round(precision_score(y_test, y_pred_iso, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_pred_iso), 4),
        'F1':        round(f1_score(y_test, y_pred_iso, zero_division=0), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_score_iso), 4),
    }

df_if = pd.DataFrame(results_if).T
print('\n=== IsolationForest con distintos contamination ===')
print(df_if.sort_values('F1', ascending=False).to_string())


In [ ]:
# Mejor IsolationForest vs Stacking
best_contam = df_if['F1'].idxmax()
contam_val = float(best_contam.split('=')[1])

iso_best = IsolationForest(contamination=contam_val, random_state=42, n_jobs=-1)
iso_best.fit(X_train_normal)
y_score_iso = -iso_best.score_samples(X_test)

# Stacking con umbral óptimo
stack = models['stacking_best']
thr   = models['stacking_best_threshold']
y_prob_stack = stack.predict_proba(X_test)[:, 1]

prec_s, rec_s, thr_s = precision_recall_curve(y_test, y_prob_stack)
f1_s = 2 * prec_s[:-1] * rec_s[:-1] / (prec_s[:-1] + rec_s[:-1] + 1e-9)
mask_s = rec_s[:-1] >= 0.85
idx_s = np.argmax(f1_s * mask_s) if mask_s.any() else np.argmax(f1_s)
y_pred_stack = (y_prob_stack >= thr_s[idx_s]).astype(int)

comp = {
    f'IsolationForest ({best_contam})': results_if[best_contam],
    'Stacking LGBM+RF (supervisado)': {
        'Precision': round(precision_score(y_test, y_pred_stack), 4),
        'Recall':    round(recall_score(y_test, y_pred_stack), 4),
        'F1':        round(f1_score(y_test, y_pred_stack), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob_stack), 4),
    }
}
print('=== No supervisado vs Supervisado ===')
print(pd.DataFrame(comp).T.to_string())

# Curvas PR comparadas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for name, yp, color in [
    ('IsolationForest', y_score_iso, 'darkorange'),
    ('Stacking (supervisado)', y_prob_stack, 'crimson'),
]:
    prec, rec, _ = precision_recall_curve(y_test, yp)
    ax.plot(rec, prec, color=color, lw=2, label=name)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Curva Precision-Recall')
ax.axvline(0.85, color='black', ls='--', lw=1, label='Recall objetivo')
ax.legend()

# Distribución scores sobre test
ax = axes[1]
for label, name, color in [(0, 'Normal', '#aec6cf'), (1, 'Fallo', '#e74c3c')]:
    mask = y_test == label
    ax.hist(y_score_iso[mask], bins=40, alpha=0.6,
            color=color, label=name, density=True)
ax.set_xlabel('Anomaly score (IsolationForest)')
ax.set_ylabel('Densidad')
ax.set_title('Distribución scores — ¿separa bien?')
ax.legend()

plt.suptitle('IsolationForest vs Stacking supervisado', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


---
## 5. Conclusiones del análisis no supervisado

### Resumen de resultados

**1. PCA — estructura del espacio de features**

| Métrica | Valor | Interpretación |
|---|---|---|
| Varianza explicada PC1+PC2 | **58.6 %** | Más de la mitad de la información cabe en 2D |
| Feature dominante en PC1 | **Torque** | El par motor es la variable más discriminante globalmente |
| Feature dominante en PC2 | **Air Temp** | La temperatura separa condiciones de operación en el segundo eje |

Los fallos aparecen parcialmente agrupados en 2D pero con solapamiento → la separabilidad es **no lineal**, lo que justifica usar árboles y stacking en lugar de Regresión Logística.

---

**2. Clustering — ¿recuperamos los 5 tipos de fallo sin etiquetas?**

| Métrica | Valor | Interpretación |
|---|---|---|
| Silhouette (k=5) | **0.293** | Clusters con solapamiento moderado — no son compactos |
| ARI vs tipos reales | **0.337** | Recuperación parcial — mejor que aleatorio pero no perfecta |

El ARI de 0.337 (escala 0–1) indica que KMeans **sí capta parte de la estructura** de los tipos de fallo, pero no los recupera completamente. Esto es esperable: RNF es completamente aleatorio (sin firma en sensores) y algunos modos como OSF y TWF comparten condiciones físicas similares (desgaste alto).

---

**3. IsolationForest vs Stacking supervisado**

| Modelo | F1 | Recall | Precision | ROC-AUC | Etiquetas necesarias |
|---|---|---|---|---|---|
| **IsolationForest** | **0.372** | 0.426 | 0.330 | 0.889 | ❌ No |
| **Stacking LGBM+RF** | **0.913** | 0.853 | 0.983 | 0.975 | ✅ Sí |
| **Diferencia** | **+0.542** (+2.5×) | +0.427 | +0.653 | +0.086 | — |

El modelo supervisado es **2.5 veces mejor en F1**. Ese es el valor concreto de tener un histórico de fallos etiquetados.

Sin embargo, IsolationForest no es inútil: con un AUC de 0.889 detecta anomalías reales con cierta fiabilidad. En un escenario real donde la máquina acaba de instalarse y no hay historial, desplegar IsolationForest desde el día 1 y sustituirlo por el supervisado conforme se acumulan etiquetas es una **estrategia de arranque válida**.

---

**Aplicación práctica recomendada:**
```
Día 0 → IsolationForest (sin etiquetas, AUC=0.889)
          ↓ acumular fallos etiquetados
~500 muestras → Reemplazar por Stacking supervisado (AUC=0.975)
```

In [ ]:
print('=== Resumen análisis no supervisado ===')
print(f'\n1. PCA: varianza explicada PC1+PC2 = {pca.explained_variance_ratio_.sum():.1%}')
print(f'   → Feature más discriminativa en PC1: {feat_names[np.argmax(abs(pca.components_[0]))]}')
print(f'   → Feature más discriminativa en PC2: {feat_names[np.argmax(abs(pca.components_[1]))]}')

best_k = list(ks)[np.argmax(sil_scores)]
print(f'\n2. Clustering: k óptimo por Silhouette = {best_k}')
print(f'   ARI con tipos reales (k=5) = {ari:.4f}')
print(f'   → {"Los clusters SÍ recuperan los tipos reales" if ari > 0.3 else "Los clusters NO recuperan bien los tipos reales"}')

f1_if   = df_if['F1'].max()
f1_sup  = round(f1_score(y_test, y_pred_stack), 4)
print(f'\n3. Detección anomalías:')
print(f'   IsolationForest F1 = {f1_if:.4f}')
print(f'   Stacking F1        = {f1_sup:.4f}')
print(f'   → El supervisado gana en +{f1_sup - f1_if:.3f} F1')
print(f'   → Pero IF no necesita etiquetas — útil si mañana aparece un tipo de fallo nuevo')
